# Weekly C Progress — Streaming Sensor Analyzer

## Final Reflection

This notebook documents my final progress on the **Streaming Sensor Analyzer** task.

The purpose of the exercise was not only to produce working C code, but to practice the way a larger C program should be designed, debugged, and broken into smaller responsibilities.

During this task I worked with:

- file input
- formatted text parsing
- string handling
- `struct`
- numeric conversion
- validation
- per-device statistics
- function decomposition
- malformed input handling
- array bounds
- streaming-style processing

The most important part of the task for me was learning how to move from a first direct implementation toward a more structured solution.

# Task Summary

The input consists of sensor records in this form:

```text
2026-09-13T08:14:31;MOTOR-03;72.4;OK
2026-09-13T08:14:32;PUMP-01;91.7;WARNING
2026-09-13T08:14:33;MOTOR-03;105.2;CRITICAL
```

The goal is to calculate statistics for each device:

```text
Device
Valid readings
Average
Minimum
Maximum
Warnings
Critical
```

The program also has to recognize malformed input without crashing.

# My Final Program Structure

By the end of the task I divided the program into several responsibilities:

```text
main()
    ↓
read line from file
    ↓
parse fields
    ↓
validate state and value
    ↓
find or create device
    ↓
update statistics
    ↓
print results
```

This was a major improvement compared with trying to perform all parsing and calculations directly inside one large block of code.

# 1. Learning to Parse Structured Text

One of the first lessons was that structured input should be treated according to its delimiters.

The records use `;` to separate fields.

Instead of manually copying individual characters based on fixed positions, I learned to parse the record according to its actual structure.

The main parsing pattern became:

```c
sscanf(
    input,
    "%19[^;];%19[^;];%19[^;];%19[^\n]",
    data.dateandTime,
    data.model,
    data.valuetext,
    data.state
);
```

This helped me understand scansets such as `%19[^;]`, meaning: read characters until `;` is encountered while respecting the destination buffer size.

# 2. Understanding Temporary Data vs Persistent Statistics

Another important concept was distinguishing between one parsed input record and accumulated information for one device.

I used one struct for the current input:

```c
typedef struct {
    char dateandTime[20];
    char model[20];
    char valuetext[20];
    char state[20];
} Data;
```

and another struct for statistics:

```c
typedef struct {
    char Device[20];
    int Valid_readings;
    float Sum;
    float Average;
    float Minimum;
    float Maximum;
    int WARNING;
    int CRITICAL;
} output;
```

This taught me that different stages of a program often need different data representations.

# 3. Validating Numbers with `strtof`

A major part of the task was handling values such as:

```text
72.4
105.2
INVALID
```

I learned that conversion and validation are not the same thing.

Using `strtof()` with an end pointer allowed me to check whether the entire string was converted:

```c
char *end;
float value = strtof(data.valuetext, &end);
```

This helped distinguish:

```text
72.4      → valid
INVALID   → invalid
72.4abc   → partially numeric, therefore malformed
```

# 4. Learning to Avoid Sentinel Values

At one point I considered representing malformed values with special numbers such as `-1.0` or `0.0`.

I learned why this is risky: either value could be a legitimate sensor reading.

The better idea is to keep validity separate from the numeric value and handle malformed records explicitly.

# 5. Finding Existing Devices

The input may contain the same device many times:

```text
MOTOR-03
PUMP-01
MOTOR-03
PUMP-01
```

I learned that the statistics array should contain one entry per **unique device**, not one entry per reading.

The program searches existing entries first and returns the index if a device already exists. If it does not exist, a new statistics entry is created.

This helped me understand the difference between the number of records and the number of unique entities.

# 6. Separating Find/Create Logic from Statistics Logic

One of the biggest design improvements was separating:

```text
find_or_create_device()
```

from:

```text
update_stats()
```

The first function answers:

> Which statistics entry belongs to this device?

The second answers:

> How does this new reading change the statistics?

This separation made the program easier to reason about and reduced the amount of logic inside `main()`.

# 7. Handling the First Reading Correctly

Minimum and maximum values introduced an important edge case.

For the first reading of a device, there is no previous minimum or maximum.

I learned to handle that explicitly:

```c
if (stat->Valid_readings == 0) {
    stat->Minimum = value;
    stat->Maximum = value;
}
```

Later readings can then be compared normally.

This was a useful example of how initialization conditions matter in stateful calculations.

# 8. Status Counters

The program distinguishes:

```text
OK
WARNING
CRITICAL
```

I used string comparison with `strcmp()` to classify states and update warning and critical counters.

This gave me more practice with C strings and also showed me why validation should happen before updating statistics.

# 9. Malformed Input Handling

The exercise deliberately includes malformed data such as:

```text
2026-09-13T08:14:34;VALVE-07;INVALID;OK
BROKEN LINE
```

I learned to treat malformed records as a normal part of the input rather than as something that should crash the program.

Different failure cases I worked through included:

- missing fields
- invalid numeric data
- partially numeric data
- invalid status text

The program keeps a malformed-record counter so these records can be reported separately.

# 10. Bugs I Encountered and Worked Through

This task exposed me to several kinds of bugs that are easy to create in C.

### Scope and lifetime problems
Early versions declared character arrays inside loops, which meant the data did not survive across iterations.

### Uninitialized memory
I encountered cases where buffers or struct fields were being inspected before they had meaningful values.

### Repeated parsing
At one point I parsed the same input line more than once unnecessarily.

### Mixing responsibilities
Some early functions both searched for devices and updated statistics.

### Duplicate devices
I initially risked creating a new statistics entry for every reading.

### First-reading min/max bugs
I learned that minimum and maximum values need special handling before any previous value exists.

### Malformed data affecting statistics
I learned that invalid input should be rejected before it reaches the statistics update stage.

### Capacity concerns
I also had to think about the maximum number of unique devices and where bounds checking belongs.

These bugs were useful because they forced me to understand not only what the code should do, but also how the program state changes over time.

# 11. Streaming Instead of Storing Every Reading

A major architectural lesson from this exercise was that the program does not need to store every sensor record.

Once a reading has updated:

```text
count
sum
minimum
maximum
warning count
critical count
```

the individual reading is no longer needed.

So the program can process:

```text
read
→ parse
→ update
→ discard
```

instead of:

```text
read everything
→ store everything
→ process later
```

This is especially important when thinking about very large files.

# 12. The 40 GB Design Lesson

The exercise also asked how the design would change if the input file became **40 GB**.

A design that reads and stores every record would require memory proportional to the number of readings:

```text
O(number of readings)
```

The streaming statistics design instead keeps memory mainly proportional to the number of unique devices:

```text
O(number of devices)
```

This showed me how an architectural decision that seems small on a test file becomes very important at larger scale.

# 13. Final Submission Code

Below is the version reached at the end of this weekly exercise.

In [ ]:
#include <string.h>
#include <stdio.h>
#include <stdbool.h>
#include <stdlib.h>
#define MAX_DATA 10000

typedef struct {
    char dateandTime[20];
    char model[20];
    char valuetext[20];
    char state[20];
} Data;

typedef struct {
    char Device[20];
    int Valid_readings;
    float Sum;
    float Average;
    float Minimum;
    float Maximum;
    int WARNING;
    int CRITICAL;
} output;

void print_stats(output stats[], int device_count, int malformed) {
    for (int i = 0; i < device_count; i++) {
        printf("Device: %s\n", stats[i].Device);
        printf("Valid readings: %d\n", stats[i].Valid_readings);
        printf("Sum: %.2f\n", stats[i].Sum);
        printf("Average: %.2f\n", stats[i].Average);
        printf("Minimum: %.2f\n", stats[i].Minimum);
        printf("Maximum: %.2f\n", stats[i].Maximum);
        printf("WARNING count: %d\n", stats[i].WARNING);
        printf("CRITICAL count: %d\n", stats[i].CRITICAL);
        printf("\n");
    }

    printf("Total malformed readings: %d\n", malformed);
}

void update_stats(output *stat, float value, const char *state) {
    if (stat->Valid_readings == 0) {
        stat->Minimum = value;
        stat->Maximum = value;
    } else {
        if (value < stat->Minimum) {
            stat->Minimum = value;
        }

        if (value > stat->Maximum) {
            stat->Maximum = value;
        }
    }

    if (strcmp(state, "WARNING") == 0) {
        stat->WARNING++;
        stat->Valid_readings++;
        stat->Sum += value;
        stat->Average = stat->Sum / stat->Valid_readings;
    } else if (strcmp(state, "CRITICAL") == 0) {
        stat->CRITICAL++;
        stat->Valid_readings++;
        stat->Sum += value;
        stat->Average = stat->Sum / stat->Valid_readings;
    } else if (strcmp(state, "OK") == 0) {
        stat->Valid_readings++;
        stat->Sum += value;
        stat->Average = stat->Sum / stat->Valid_readings;
    }
}

int find_or_create_device(output stats[], int *device_count, const char *model) {
    if (device_count == NULL || model == NULL) {
        return -1;
    } else {
        for (int i = 0; i < *device_count; i++) {
            if (strcmp(stats[i].Device, model) == 0) {
                return i;
            }
        }

        if (*device_count >= MAX_DATA) {
            return -1;
        }

        strcpy(stats[*device_count].Device, model);
        stats[*device_count].Valid_readings = 0;
        stats[*device_count].Sum = 0.0f;
        stats[*device_count].Average = 0.0f;
        stats[*device_count].WARNING = 0;
        stats[*device_count].CRITICAL = 0;

        (*device_count)++;
        return *device_count - 1;
    }
}

int main() {
    int i = 0;
    char input[10000];
    char askedmodel[20];
    output stats[MAX_DATA];
    Data data;
    FILE *file = fopen("read.txt", "r");
    bool value_valid;
    int valid = 0;
    int malformed = 0;
    int device_count = 0;
    int index = 0;

    if (file == NULL) {
        return 1;
    }

    while (fgets(input, sizeof input, file) != NULL) {
        int fields = sscanf(
            input,
            "%19[^;];%19[^;];%19[^;];%19[^\n]",
            data.dateandTime,
            data.model,
            data.valuetext,
            data.state
        );

        if (fields == 4) {
            char *end;
            float value = strtof(data.valuetext, &end);

            bool state_valid =
                strcmp(data.state, "OK") == 0 ||
                strcmp(data.state, "WARNING") == 0 ||
                strcmp(data.state, "CRITICAL") == 0;

            if (!state_valid) {
                malformed++;
                continue;
            }

            if (end == data.valuetext || *end != '\0') {
                value_valid = false;
            } else {
                value_valid = true;
            }

            if (value_valid) {
                index = find_or_create_device(stats, &device_count, data.model);

                if (index != -1) {
                    update_stats(&stats[index], value, data.state);
                }

                i++;
            } else {
                malformed++;
            }
        } else if (fields != 4) {
            malformed++;
            continue;
        }
    }

    fclose(file);
    print_stats(stats, device_count, malformed);

    return 0;
}

# 14. What I Improved During This Exercise

Comparing the beginning of the task with the final version, I improved in several areas.

### C syntax and memory thinking
I became more comfortable with arrays, pointers, struct fields, string comparison, and function arguments.

### Debugging
Instead of only looking at whether the final output was correct, I started tracing:

```text
what does this variable mean?
when is it initialized?
who modifies it?
what happens on the first iteration?
what happens on malformed input?
```

### Program architecture
I moved from one large block of logic toward parsing, validation, device lookup, statistics, and reporting as separate concerns.

### Data processing
I learned the difference between keeping raw data and keeping only the information needed for the final result.

### Defensive programming
The malformed input cases made me think more carefully about unexpected values and array limits.

# 15. Main Takeaway

The most important lesson from this weekly task was that solving a programming problem is not only about finding the correct syntax.

A large part of the work was deciding:

```text
what information should exist?
where should it be stored?
when should it be validated?
which function should be responsible for it?
what information can be discarded?
```

The program became easier to understand as those responsibilities became clearer.

This exercise gave me practical experience with how a C program can evolve through debugging, refactoring, and repeated testing rather than being written correctly in one attempt.

# Weekly Progress Status

**Topics practiced:**

- C file I/O
- `fgets`
- `sscanf`
- scansets
- `strtof`
- pointers
- strings
- `strcmp`
- structs
- arrays of structs
- statistics accumulation
- malformed-record handling
- function decomposition
- bounds checking
- streaming algorithms
- memory-complexity reasoning

This task is part of my ongoing weekly programming practice aimed at improving problem-solving ability, C programming habits, and software-design skills.